# Why identical fill-ins get different probabilities

In `traceletReAct_trajectory.ipynb`, step 2 of question 4 showed three candidates that emitted
**byte-identical code** yet were assigned different fill-in probabilities:

| cand | sum logprob | P(fill-in) | tok |
|---|---|---|---|
| c0 | -0.0061 | 0.993938 | 13 |
| c1 | -0.0148 | 0.985284 | 13 |
| c2 | -0.0092 | 0.990856 | 13 |

`P(fill-in)` is `exp(sum logprob)`, so the question is why the same 13-token sequence sums to
-0.0061 in one choice and -0.0148 in another.

The answer is that **scoring is not deterministic across positions within one `n>1` request**. The
same token, with the same prefix, gets a different logprob depending on which other completions
share its batch.

Two features of the data say this is *not* mere floating-point rounding:

- **The magnitudes are far too large.** Cell 4 below shows a single token differing by ~0.29 in
  logprob, and across the completed sweeps the p95 spread in summed logprob is 0.27 with a maximum
  of 4.4 — a 79x ratio once exponentiated.
- **It clusters.** Choices do not each drift by a little; they fall into groups that agree
  bit-for-bit, with the groups differing from each other. Low-bit rounding would not do that.

The established cause is **lack of batch invariance in the kernels themselves**, not anything
model-specific. Matmul, RMSNorm and attention are all reduction operations, and GPU kernels pick a
different reduction *schedule* depending on the batch size, to maximise throughput. A different
reduction order gives a different rounding path, so the same token's logits change with the shape
of the batch it happens to land in. See Thinking Machines Lab,
[Defeating Nondeterminism in LLM Inference](https://thinkingmachines.ai/blog/defeating-nondeterminism-in-llm-inference/).

**This is not specific to hosted APIs.** It reproduces on open-weight models served locally: under
default vLLM, Qwen3-8B at temperature 0 produced 80 distinct completions across 1000 identical
requests. vLLM now ships a fix -- `VLLM_BATCH_INVARIANT=1` -- which swaps in batch-invariant
RMSNorm/GEMM/attention kernels and restores bit-identical output (1000/1000), at a 1.6-2.1x latency
cost. It requires NVIDIA compute capability 8.0+ and is tested across dense and MoE architectures
(Qwen3, DeepSeek, Llama 3, Mistral, Phi, Granite). vLLM's own announcement reports that logprobs
come out exactly equal at bs=1 and bs=N with the flag on, though the docs page does not state
logprob coverage explicitly.

That matters for this project: running fill-in sampling against a local vLLM server with batch
invariance enabled would make `P(fill-in)` exactly reproducible, which is a precondition for
treating it as anything more than a diagnostic.

This notebook demonstrates it live, then measures how often it happens in the completed sweeps.

Ruled out from the recorded data: the `-9999` placeholder bug (`n_placeholder` is 0 in all 770
affected groups), different tokenization (token counts match exactly), and display truncation
(the code strings are byte-equal).

In [7]:
import os
import sys
from collections import defaultdict

sys.path.insert(0, "../examples/open_deep_research")

from dotenv import load_dotenv

load_dotenv()

from smolagents import OpenAIModel
from smolagents.models import ChatMessage, MessageRole

MODEL = "gpt-4o"
N = 5           # candidates per request, as in a tracelet fill-in call
MAX_TRIES = 6   # keep sampling until two choices come back identical

model = OpenAIModel(model_id=MODEL, api_key=os.environ["OPENAI_API_KEY"])

# Shaped like a real fill-in call, and deliberately low-entropy so duplicate
# completions are likely -- which is exactly the case that exposes the effect.
PROMPT = '''Fill in the sentinel below. Reply with exactly one line of the form 'ARG0: <value>'.

Code skeleton:
    moon_text = visit_page(url=ARG0)

Task: open the English Wikipedia article about the Moon.

ARG0:'''

messages = [ChatMessage(role=MessageRole.USER, content=[{"type": "text", "text": PROMPT}])]

In [9]:
# Sample until some token sequence comes back more than once in a single request.
duplicate = None
for attempt in range(1, MAX_TRIES + 1):
    response = model.generate(messages, n=N, logprobs=True, top_logprobs=5)
    choices = response.raw.choices

    groups = defaultdict(list)
    for i, ch in enumerate(choices):
        tokens = tuple(t.token for t in ch.logprobs.content)
        groups[tokens].append(i)

    print(f"attempt {attempt}: {len(choices)} choices, {len(groups)} distinct token sequences")
    for i, ch in enumerate(choices):
        total = sum(t.logprob for t in ch.logprobs.content)
        print(f"   choice {i}: {len(ch.logprobs.content):>3} tok  sum={total:+.10f}  {ch.message.content!r}")

    repeated = {toks: idxs for toks, idxs in groups.items() if len(idxs) > 1}
    if repeated:
        tokens, idxs = max(repeated.items(), key=lambda kv: len(kv[1]))
        duplicate = (choices, tokens, idxs)
        print(f"\n-> {len(idxs)} choices share an identical {len(tokens)}-token sequence: {idxs}")
        break
else:
    print("\nNo duplicate in this run -- re-run the cell, or lower the entropy of PROMPT further.")

attempt 1: 5 choices, 3 distinct token sequences
   choice 0:  13 tok  sum=-1.9454292735  "ARG0: 'https://en.wikipedia.org/wiki/Moon'"
   choice 1:  13 tok  sum=-0.6976527229  'ARG0: "https://en.wikipedia.org/wiki/Moon"'
   choice 2:  13 tok  sum=-0.6976331735  'ARG0: "https://en.wikipedia.org/wiki/Moon"'
   choice 3:  11 tok  sum=-0.8538891041  'ARG0: https://en.wikipedia.org/wiki/Moon'
   choice 4:  11 tok  sum=-0.8538891041  'ARG0: https://en.wikipedia.org/wiki/Moon'

-> 2 choices share an identical 13-token sequence: [1, 2]


In [10]:
# The point: identical text, identical tokens, different summed logprob.
assert duplicate, "no duplicate found -- re-run the previous cell"
choices, tokens, idxs = duplicate

print(f"text (identical across choices {idxs}):\n  {''.join(tokens)!r}\n")
for i in idxs:
    content = choices[i].logprobs.content
    total = sum(t.logprob for t in content)
    print(f"  choice {i}: {len(content)} tokens, sum={total:+.10f}, P(fill-in)=exp(sum)={pow(2.718281828459045, total):.6f}")

sums = [sum(t.logprob for t in choices[i].logprobs.content) for i in idxs]
print(f"\nspread in summed logprob : {max(sums) - min(sums):.10f}")
print(f"spread per token          : {(max(sums) - min(sums)) / len(tokens):.10f}")
print(f"identical logprobs?       : {'YES (no effect this time)' if len(set(sums)) == 1 else 'NO -- the effect'}")

text (identical across choices [1, 2]):
  'ARG0: "https://en.wikipedia.org/wiki/Moon"'

  choice 1: 13 tokens, sum=-0.6976527229, P(fill-in)=exp(sum)=0.497752
  choice 2: 13 tokens, sum=-0.6976331735, P(fill-in)=exp(sum)=0.497762

spread in summed logprob : 0.0000195494
spread per token          : 0.0000015038
identical logprobs?       : NO -- the effect


In [5]:
# Where the divergence starts: early tokens usually agree bit-for-bit, then drift.
columns = [choices[i].logprobs.content for i in idxs]
print(f"{'tok':>4} {'token':>20} " + " ".join(f"{f'choice {i}':>14}" for i in idxs) + "   spread")
for t in range(len(columns[0])):
    values = [c[t].logprob for c in columns]
    spread = max(values) - min(values)
    print(
        f"{t:>4} {columns[0][t].token!r:>20} "
        + " ".join(f"{v:>+14.8f}" for v in values)
        + ("" if spread == 0 else f"   {spread:.2e}")
    )

 tok                token       choice 1       choice 2       choice 3       choice 4   spread
   0                'ARG'    -0.00320385    -0.00320385    -0.00320385    -0.00320385
   1                  '0'    +0.00000000    +0.00000000    +0.00000000    +0.00000000
   2                  ':'    +0.00000000    +0.00000000    +0.00000000    +0.00000000
   3                 ' "'    -0.13313219    -0.13313219    -0.13313219    -0.13313219
   4              'https'    -0.00000067    -0.00000031    -0.00000067    -0.00000031   3.58e-07
   5                '://'    +0.00000000    +0.00000000    +0.00000000    +0.00000000
   6                 'en'    +0.00000000    +0.00000000    +0.00000000    +0.00000000
   7         '.wikipedia'    -0.00002296    -0.00001629    -0.00002296    -0.00001629   6.68e-06
   8               '.org'    +0.00000000    +0.00000000    +0.00000000    +0.00000000
   9              '/wiki'    -0.00000043    -0.00000043    -0.00000043    -0.00000043
  10                 '/

## How often this happens in the completed sweeps

Every candidate record stores the assembled `code` plus the fill-in logprob, so the recorded runs
can be scanned for the same signature: two candidates in one step whose code is byte-identical but
whose logprobs differ. If token counts always match across such a group, tokenization is excluded
and one cause is left.

In [11]:
import pickle
from pathlib import Path

SWEEPS = [
    "tracelet_direct_tp_v6n3_react_gpt-4o",
    "tracelet_direct_n5_tp_v6n5_react_gpt-4o",
    "tracelet_direct_tp_v6n3_react_gpt-5.4-mini",
]

groups_found, clean = 0, []
for sweep in SWEEPS:
    for f in Path(sweep).glob("*.pkl"):
        try:
            record = pickle.load(open(f, "rb"))
        except Exception:
            continue
        for step in record["steps"]:
            by_code = defaultdict(list)
            for c in step.get("candidates") or []:
                by_code[c.get("code")].append(c)
            for group in by_code.values():
                logprobs = [c.get("fillin_logprob") for c in group]
                if len(group) < 2 or None in logprobs or len(set(logprobs)) == 1:
                    continue
                groups_found += 1
                # Keep only groups where tokenization is excluded as an explanation, and where
                # no -9999 placeholder was filtered out of the sum.
                if len({c.get("fillin_n_tokens") for c in group}) != 1:
                    continue
                if max((c.get("fillin_n_placeholder") or 0) for c in group) != 0:
                    continue
                clean.append((max(logprobs) - min(logprobs), group[0].get("fillin_n_tokens")))

print(f"candidate groups with identical code but differing logprobs : {groups_found}")
print(f"  ...with matching token counts and no placeholders         : {len(clean)}")
print(f"  ...excluded (token counts differ, so tokenization could explain them) : {groups_found - len(clean)}")

if clean:
    import statistics

    absolute = sorted(s for s, _ in clean)
    per_token = sorted(s / n for s, n in clean if n)
    p95 = absolute[int(len(absolute) * 0.95)]
    print(f"\nspread in summed logprob : median {statistics.median(absolute):.5f}, p95 {p95:.5f}, max {max(absolute):.5f}")
    print(f"spread per token          : median {statistics.median(per_token):.7f}, max {max(per_token):.7f}")
    print(f"\nworst case exponentiated : identical fill-ins differing by {max(absolute):.2f} in summed")
    print(f"logprob get P(fill-in) values a factor of {pow(2.718281828459045, max(absolute)):.0f}x apart.")
    print("\nBecause P(fill-in) is exp(SUM), the discrepancy compounds with fill-in length, so long")
    print("fill-ins carry a larger spurious probability difference than short ones.")

candidate groups with identical code but differing logprobs : 876
  ...with matching token counts and no placeholders         : 770
  ...excluded (token counts differ, so tokenization could explain them) : 106

spread in summed logprob : median 0.01206, p95 0.26642, max 4.37500
spread per token          : median 0.0007424, max 0.3012354

worst case exponentiated : identical fill-ins differing by 4.37 in summed
logprob get P(fill-in) values a factor of 79x apart.

Because P(fill-in) is exp(SUM), the discrepancy compounds with fill-in length, so long
fill-ins carry a larger spurious probability difference than short ones.
